# Feature Engineering

The following features are engineered based on insights derived from the Exploratory Data Analysis (EDA). The EDA revealed patterns related to delayed congestion effects, modal evacuation imbalance, and seasonal operational stress. These observations motivate the creation of additional variables that better capture the underlying operational dynamics affecting import dwell time.



## Load the cleaned dataset 


In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("../../Data/JNPA_Cleaned_Data.csv")
df["Date"] = pd.to_datetime(df["Date"])

df_f = df.copy()

### Engineering Features 

#### 1.Volume_Lag1:


Temporal analysis in the EDA indicated that increases in container throughput are occasionally followed by higher import dwell times in the following month. This suggests a delayed congestion effect where containers accumulate in the yard before evacuation pressure affects dwell performance.

To capture this delayed effect, a lagged throughput variable was engineered representing the container volume handled in the previous month.

##### Volume_Lag1 = Total_TEUs_Handled(t−1)

In [2]:
df_f["Volume_Lag1"] = df_f["Total_TEUs_Handled"].shift(1)

#### 2.Rail_Friction:

The EDA showed that train and truck dwell times exhibit weak correlation, indicating that delays in one evacuation mode do not necessarily affect the other. This suggests that rail and truck evacuation systems can experience independent operational bottlenecks.

To capture the relative performance between the two evacuation modes, a rail friction metric was introduced. This variable measures the ratio between train dwell time and truck dwell time.
##### Rail_Friction = Train_Dwell / Truck_Dwell

In [3]:
df_f["Rail_Friction"] = df_f["Imp_Dwell_Train"] / df_f["Imp_Dwell_Truck"]


#### 3.Stress:

Temporal analysis highlighted increased volatility in dwell times during monsoon months, suggesting that environmental disruptions may interact with throughput pressure to affect evacuation efficiency.

To capture this interaction, a seasonal stress feature was engineered by combining container throughput with a monsoon indicator variable.
#### Stress = Total_TEUs_Handled × Is_Monsoon

In [4]:
df_f["Is_Monsoon"] = df_f["Date"].dt.month.isin([6,7,8,9]).astype(int)

df_f["Stress"] = df_f["Total_TEUs_Handled"] * df_f["Is_Monsoon"]

#### Risk:




Port demurrage charges are typically triggered when container dwell time exceeds the free storage period allowed by the terminal. At JNPA terminals, the free time for import containers is approximately 72 hours after vessel discharge. However, the dataset used in this study contains monthly average dwell times rather than individual container dwell observations.

Because averages mask the distribution of individual containers, directly modeling the 72-hour demurrage boundary would underestimate congestion risk. Instead, a data-driven threshold was adopted to identify unusually high dwell conditions within the dataset.

The congestion risk label was therefore defined using the upper quartile of import dwell time:

Risk = 1 if Import_Dwell_Overall > Q3  
Risk = 0 otherwise

This approach identifies the top 25% of months with the highest dwell times as congestion periods, allowing the model to detect operational conditions associated with elevated demurrage exposure.

In [5]:
# Calculate the 75th percentile threshold
risk_threshold = df_f["Imp_Dwell_Overall"].quantile(0.75)

print("New Risk Threshold (Q3):", risk_threshold)

# Create Risk label
df_f["Risk"] = (df_f["Imp_Dwell_Overall"] > risk_threshold).astype(int)


New Risk Threshold (Q3): 27.15


#### Save Feature Engineered dataset

In [6]:
df_f = df_f.dropna()
df_f.to_csv("../../Data/JNPA_feature_engineered.csv", index=False)


In [7]:
# Check how many observations fall into each class
print(df_f["Risk"].value_counts())

# Check class proportions
print(df_f["Risk"].value_counts(normalize=True))

Risk
0    25
1     9
Name: count, dtype: int64
Risk
0    0.735294
1    0.264706
Name: proportion, dtype: float64
